

# 01_data_ingestion.ipynb: Panel Construction & Data Engineering

##  Overview
This notebook serves as the foundational data engineering pipeline for the project. It seamlessly integrates raw financial datasets (CSMAR), third-party asset pricing factors (CH4, FF3, q5), and pre-computed Natural Language Processing (NLP) sentiment features. Through rigorous data cleaning, extreme memory optimization, and strict time-alignment mechanisms, it fuses unstructured and structured data into a comprehensive, high-frequency **Daily Master Panel** ready for empirical asset pricing analysis.

---

## Architectural Features

1. **High-Performance I/O (Parquet Engine)**
   - Completely deprecates slow Excel/CSV I/O operations. The pipeline features an automated recursive scanner that penetrates the `data/raw/` directory, strips out CSMAR-specific redundant header rows, standardizes stock tickers (6-digit strings), and converts all raw data into column-oriented `.parquet` files, accelerating downstream read speeds by over 100x.

2. **Strict Anti-Look-Ahead Bias Mechanisms**
   - Implements temporal alignment using `pd.merge_asof`. Financial fundamentals (e.g., Balance Sheets, Income Statements) are strictly lagged by a standard 4-month reporting window, while Earnings Surprises (SUE) are matched using exact historical announcement dates to ensure zero forward-looking bias in predictive regressions.

3. **Intelligent Market-Type Filtering**
   - Automatically detects and resolves multi-market redundancy issues inherent in CSMAR factor datasets. It extracts the cleanest time series based on established academic hierarchies (prioritizing `P9714`: A-shares + ChiNext + STAR Market).

4. **Extreme Memory Optimization & Hashing**
   - Engineered to handle 12+ million firm-day observations locally. It utilizes downcasting (`float64` to `float32`), dynamic garbage collection (`gc.collect()`), and pure numeric Hash-Mapping for time-series factor merging, keeping RAM footprint strictly minimized.

---

## Data Dependencies

Before executing this pipeline, ensure your local `data/raw/` directory is populated with the required datasets (the recursive scanner `rglob` will automatically locate them across any subdirectories):

```text
data/raw/
  ├── Market & Microstructure/      # Daily returns, Monthly features (Turnover, Illiqd), Limit-hit status, Risk-free rate
  ├── Fundamentals & Governance/    # Balance sheets, Income statements, Valuation ratios, Board characteristics
  ├── Asset Pricing Factors/        # CH4, q5, Fama-French 3-Factor models
  ├── Macro Sentiment Index/        # BW_Sentiment_Index (Baker & Wurgler, 2006)
  └── QnA NLP Data/                 # Pre-computed NLP panel (e.g., QnA_Slim_Panel.parquet)
```

---

##  Output Deliverables

Upon successful execution, the pipeline automatically generates the unified dataset in the `data/processed/` directory:

- `01_Base_Daily_Panel_Advanced.parquet`
  - **Dimensions**: ~12,000,000+ rows × 50+ columns.
  - **Contents**: A perfectly aligned spine comprising daily excess returns, NLP sentiment signals (Investor Tone, Manager Tone, ML-Substantiveness), microstructure frictions, lagged financial controls, and macroeconomic factors.
  - **Usage**: Serves as the single source of truth for downstream statistical summaries (`02`), portfolio sorting (`03`), and predictability regressions (`04` & `05`).

---

##  Execution Guide

1. Ensure the Python environment is configured with `pandas`, `numpy`, `pyarrow`, `fastparquet`, and `tqdm`.
2. Verify that the `DATA_BASE_PATH` global variable correctly points to your local data root directory.
3. Click **Run All** to initiate the pipeline.


## 0. Setup & Configurations

In [1]:
import os
import glob
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
from tqdm.auto import tqdm
import sys

current_dir = Path.cwd()
project_root = next((p for p in [current_dir] + list(current_dir.parents) if (p / 'src').exists()), None)

if project_root and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


from src.data.loader import load_optimized_parquet
from src.data.loader import safe_merge
from src.data.loader import clean_and_convert_csmar
from src.data.loader import batch_convert_raw_data
from src.data.loader import safe_load_fin
from src.data.loader import map_time_series_factor


warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
warnings.filterwarnings('ignore', category=FutureWarning)


DATA_BASE_PATH = Path(r"D:\iip_asset_pricing\data")

PATHS = {
    'raw': DATA_BASE_PATH / 'raw',              
    'processed': DATA_BASE_PATH / 'processed',  
    'results': DATA_BASE_PATH / 'results',     
}

for name, path in PATHS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"✅ 目录已就绪: {path}")

print("环境初始化完成！")

✅ 目录已就绪: D:\iip_asset_pricing\data\raw
✅ 目录已就绪: D:\iip_asset_pricing\data\processed
✅ 目录已就绪: D:\iip_asset_pricing\data\results
环境初始化完成！


## 1. Excel to Parquet

In [2]:
batch_convert_raw_data(PATHS['raw'])

========== ⚙️ 开始扫描并转换原始数据 ==========


Converting to Parquet:   0%|          | 0/152 [00:00<?, ?it/s]


========== 📊 转换任务摘要 ==========
总文件数: 152
✅ 成功转换: 0
⏩ 跳过(已存在): 152
❌ 转换失败: 0


## 2: The Daily Trading Data

 Provides the core dependent variables (returns) and other trading proxy.

| Final Variable | CSMAR Original Field | Data Type & Code Logic | Description |
| :--- | :--- | :--- | :--- |
| **Dretwd** | `Dretwd` | `Float32` | **Daily Return (Reinvested)**. Stock return considering cash dividend reinvestment. This is the primary dependent variable (LHS) for empirical asset pricing. |
| **Dretnd** | `Dretnd` | `Float32` | **Daily Return (Excluding Dividends)**. Stock return without cash dividend reinvestment. Used as a robustness check alternative. |
| **Dsmvtll** | `Dsmvtll` | `Float32` | **Daily Total Market Capitalization**. Calculated as (Total outstanding shares × Closing price). <br> *Note: The raw unit in CSMAR is in **Thousands of RMB (RMB 1,000)**.* |
| **OpenPrice** | `OpenPrice` | `Float32` | **Opening Price (Backward-Adjusted)**. Adjusted for stock splits/dividends to prevent artificial price jumps. |
| **ClosePrice** | `ClosePrice` | `Float32` | **Close Price (Backward-Adjusted)**. Adjusted for stock splits/dividends to prevent artificial price jumps. |
| **StateCode** | `StateCode` | `Int32` | **Trading Status**. Indicates if the stock is actively traded. `0` = Normal Trading, `1` = Suspended (Halted), `2` = Holiday. Used to filter out halted days. |
| **LimitStatus** | `LimitStatus` | `Float32` | **Price Limit Status**. Identifies whether the stock hit the daily price limit at close. `1` = Up-limit (涨停), `-1` = Down-limit (跌停), `0` = Normal. Critical for filtering illiquid limit-hit samples. |
| **Rf_Daily** | `Nrrdaydt` | `Float32`<br>*Logic*: `Nrrdaydt / 100 / 360` | **Daily Risk-Free Rate**. Derived from the 1-year lump-sum deposit rate (`NRI01`). <br> *Unit Correction*: Scaled down by `100` (percentage to decimal) and `360` (annualized to daily) to match the daily stock returns. |
| **Rm_Daily** | `Cdretwdos` | `Float32`<br>*Logic*: Filter `Markettype == 5 or 21` | **Value-Weighted Market Return**. Comprehensive daily market return (reinvested). `21` represents the aggregate index for SSE & SZSE A-shares + GEM, strictly excluding B-shares. |

***


In [4]:
# ==============================================================================
# 2. 核心主干：构建日度交易日历与行情特征 
# ==============================================================================
import gc
import pandas as pd

def load_optimized_parquet(keyword, usecols=None, rename_dict=None):
    """
    极限省内存读取器：读入 -> 降维 -> 清理
    支持穿透匹配【文件夹名称】与【文件名称】
    """
    # 获取 raw 目录下所有的 parquet 文件
    all_files = list(PATHS['raw'].rglob("*.parquet"))

    # 只要文件路径（包含外层文件夹名）中含有 keyword，就选中它
    files = [f for f in all_files if keyword in str(f)]

    if not files:
        print(f"   [警告] 未在任何路径中找到包含 '{keyword}' 的文件。")
        return pd.DataFrame()

    df_list = []
    for f in files:
        try:
            # 利用 Parquet 的列裁剪特性，按需加载，极省内存
            df = pd.read_parquet(f, columns=usecols)

            if rename_dict:
                df = df.rename(columns=rename_dict)

            # 1. 内存优化：日期转 datetime
            if 'Date' in df.columns:
                df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

            # 2. 内存优化：股票代码转 int32 (消灭极占内存的 String)
            if 'Stkcd' in df.columns:
                df['Stkcd'] = pd.to_numeric(df['Stkcd'], errors='coerce')
                df = df.dropna(subset=['Stkcd'])
                df['Stkcd'] = df['Stkcd'].astype('int32') # 用 int32 关联，速度极快

            # 3. 内存优化：浮点数降维 float64 -> float32
            float_cols = df.select_dtypes(include=['float64']).columns
            df[float_cols] = df[float_cols].astype('float32')

            df_list.append(df)

        except ValueError:
            # 静默拦截 CSMAR 命名混乱导致的脏文件 (例如包含LimitStatus但名字叫TRD_Dalyr的文件)
            pass
        except Exception as e:
            print(f"   [报错] 读取 {f.name} 时出错: {e}")

    if not df_list:
        return pd.DataFrame()

    df_concat = pd.concat(df_list, ignore_index=True)
    del df_list
    gc.collect() # 回收临时列表

    return df_concat

def safe_merge(df_left, df_right, on_keys, how='left'):
    """
    🛡️ 安全合并函数：防止重复运行产生 _x, _y 后缀。
    逻辑：在合并前，剔除左表中与右表同名的非键列，实现"覆盖更新"。
    """
    if df_right.empty: return df_left
    overlap_cols = [c for c in df_right.columns if c in df_left.columns and c not in on_keys]
    if overlap_cols:
        df_left = df_left.drop(columns=overlap_cols)
    return df_left.merge(df_right, on=on_keys, how=how)

In [5]:
print("========== 📥 Step 2: Assembling The Daily Spine (Memory Optimized) ==========")

# ------------------------------------------------------------------------------
# 2.1 加载日个股回报率 (作为主干左表)
# ------------------------------------------------------------------------------
print(" -> 1. 加载日个股回报率 (TRD_Dalyr)...")
cols_ret = ['Stkcd', 'Trddt', 'Dretwd', 'Dretnd', 'Dsmvtll']
df_spine = load_optimized_parquet('回报率', usecols=cols_ret, rename_dict={'Trddt': 'Date'})
if df_spine.empty:
    # 容错：如果文件夹没叫"回报率"，尝试用英文表名匹配
    df_spine = load_optimized_parquet('TRD_Dalyr', usecols=cols_ret, rename_dict={'Trddt': 'Date'})

df_spine = df_spine.dropna(subset=['Stkcd', 'Date']).drop_duplicates(subset=['Stkcd', 'Date'])
print(f"    ✅ 主干建立，样本量: {len(df_spine):,}，当前内存: {df_spine.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ------------------------------------------------------------------------------
# ------------------------------------------------------------------------------
# 2.2 加载后复权行情 (OpenPrice, ClosePrice, StateCode) 
# ------------------------------------------------------------------------------
print(" -> 2. 附加后复权行情 (OpenPrice, ClosePrice, StateCode)...")


cols_quote = ['Stkcd', 'TradingDate', 'OpenPrice', 'ClosePrice', 'StateCode'] 


df_quote = load_optimized_parquet('Quotation', usecols=cols_quote, rename_dict={'TradingDate': 'Date'})

if not df_quote.empty:
    df_quote = df_quote.drop_duplicates(subset=['Stkcd', 'Date'])
    df_spine = safe_merge(df_spine, df_quote, on_keys=['Stkcd', 'Date']) # 🛡️ 安全合并
    print(f"    ✅ 行情数据接入成功，有效匹配: {df_spine['ClosePrice'].notna().sum():,} 行")
else:
    print("    ⚠️ 警告：未能成功加载行情数据。")

del df_quote
gc.collect()

# ------------------------------------------------------------------------------
# 2.3 加载涨跌停状态 (LimitStatus)
# ------------------------------------------------------------------------------
print(" -> 3. 附加涨跌停状态 (LimitStatus)...")
cols_limit = ['Stkcd', 'Trddt', 'LimitStatus']
df_limit = load_optimized_parquet('涨跌停', usecols=cols_limit, rename_dict={'Trddt': 'Date'})

if not df_limit.empty:
    df_limit = df_limit.drop_duplicates(subset=['Stkcd', 'Date'])
    df_spine = safe_merge(df_spine, df_limit, on_keys=['Stkcd', 'Date']) # 🛡️ 安全合并
del df_limit
gc.collect()

# ------------------------------------------------------------------------------
# 2.4 加载无风险利率 (TRD_Nrrate)
# ------------------------------------------------------------------------------
print(" -> 4. 附加无风险利率 (Rf_Daily)...")
cols_rf = ['Clsdt', 'Nrrdaydt']
df_rf = load_optimized_parquet('无风险利率', usecols=cols_rf, rename_dict={'Clsdt': 'Date', 'Nrrdaydt': 'Rf_Daily'})

if not df_rf.empty:
    df_rf['Rf_Daily'] = df_rf['Rf_Daily'] / 100.0 / 360.0
    df_rf = df_rf.dropna(subset=['Date']).drop_duplicates(subset=['Date'])
    df_spine = safe_merge(df_spine, df_rf, on_keys=['Date']) # 🛡️ 安全合并
del df_rf
gc.collect()

# ------------------------------------------------------------------------------
# 2.5 加载大盘基准收益率 (TRD_Cndalym)
# ------------------------------------------------------------------------------
print(" -> 5. 附加综合市场回报率 (Rm_Daily)...")
cols_mkt = ['Markettype', 'Trddt', 'Cdretwdos']
df_mkt = load_optimized_parquet('市场回报', usecols=cols_mkt, rename_dict={'Trddt': 'Date', 'Cdretwdos': 'Rm_Daily'})

if not df_mkt.empty:
    df_mkt = df_mkt[df_mkt['Markettype'].isin([5, 21])].copy()
    df_mkt = df_mkt.dropna(subset=['Date']).drop_duplicates(subset=['Date'])[['Date', 'Rm_Daily']]
    df_spine = safe_merge(df_spine, df_mkt, on_keys=['Date']) # 🛡️ 安全合并
del df_mkt
gc.collect()

# ------------------------------------------------------------------------------
# 2.6 收尾：还原 Stkcd 为 6 位字符串
# ------------------------------------------------------------------------------
print(" -> 6. 收尾：还原 Stkcd 格式...")
# int32 的使命结束了，转回我们熟悉的 '000001'
df_spine['Stkcd'] = df_spine['Stkcd'].astype(str).str.zfill(6)

print("\n🚀 核心交易主干构建完成！")
print(f"最终数据维度: {df_spine.shape}")
print(f"最终占用内存: {df_spine.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# 预览：只展示完全匹配到所有行情数据的有效行，避免看到满眼 NaN
print("\n👀 数据预览 (展示成功匹配行情数据的记录):")
valid_spine = df_spine.dropna(subset=['OpenPrice', 'ClosePrice','LimitStatus'])
if not valid_spine.empty:
    display(valid_spine[['Stkcd', 'Date', 'Dretwd', 'OpenPrice', 'ClosePrice', 'LimitStatus']].head(3))
else:
    display(df_spine.head(3))

========== 📥 Step 2: Assembling The Daily Spine (Memory Optimized) ==========
 -> 1. 加载日个股回报率 (TRD_Dalyr)...
    ✅ 主干建立，样本量: 12,386,319，当前内存: 283.50 MB
 -> 2. 附加后复权行情 (OpenPrice, ClosePrice, StateCode)...
    ✅ 行情数据接入成功，有效匹配: 12,386,296 行
 -> 3. 附加涨跌停状态 (LimitStatus)...
 -> 4. 附加无风险利率 (Rf_Daily)...
 -> 5. 附加综合市场回报率 (Rm_Daily)...
 -> 6. 收尾：还原 Stkcd 格式...

🚀 核心交易主干构建完成！
最终数据维度: (12386319, 11)
最终占用内存: 1216.69 MB

👀 数据预览 (展示成功匹配行情数据的记录):


,Stkcd,Date,Dretwd,OpenPrice,ClosePrice,LimitStatus
1000000,002413,2012-07-02,0.022201,15.803,16.153999,0.0
1000001,002413,2012-07-03,-0.052880,16.170,15.300000,0.0
1000002,002413,2012-07-04,-0.029910,15.331,14.842000,0.0


In [30]:
df_quote = load_optimized_parquet('Quotation', usecols=cols_quote, rename_dict={'Symbol': 'Stkcd', 'TradingDate': 'Date'})

## 3. Interactive & NLP Signals


**Design Intent**: To integrate high-frequency textual signals into the daily trading spine. This block merges two critical sources of unstructured data: the interactive Q&A platform (providing our core independent variables) and financial news (providing essential media controls).

**Feature Engineering Highlight**: For financial news, we calculate backward-rolling features (e.g., `Newstone_t_30_t_3`) directly in this data ingestion pipeline. Computing a 20-trading-day rolling mean shifted by 3 days on a 12-million-row panel is computationally expensive. Pre-computing it here saves massive I/O and CPU time during downstream Panel OLS regressions, while strictly preventing look-ahead bias.

| Final Variable | Original Field / Source | Data Type & Code Logic | Description |
| :--- | :--- | :--- | :--- |
| **Investor_Tone** | `Investor_Tone` (QnA Panel) | `Float32` | **Retail Investor Tone**. Sentiment score extracted from investors' questions. |
| **Manager_Tone** | `Manager_Tone` (QnA Panel) | `Float32` | **Managerial Tone**. Sentiment score extracted from executives' replies. |
| **Substantiveness_ML** | `Substantiveness_ML` (QnA Panel) | `Float32` | **Reply Substantiveness**. Machine-learning derived probability indicating whether the manager's reply contains hard, substantive information rather than evasive boilerplate. |
| **Num_Questions** | `Num_Questions` (QnA Panel) | `Float32` | **Interaction Volume**. Daily number of questions asked by investors. Missing values are filled with `0`. |
| **Ln_Num_Questions** | Derived | `Float32`<br>*Logic*: `Ln(1 + Num_Questions)` | **Log Attention**. Log-transformed interaction volume to proxy for retail investor attention. |
| **News_Dummy** | `Newsid` (News) | `Int8` (Dummy) | **News Coverage Indicator**. 1 if the firm has financial news coverage on day $t$, 0 otherwise. |
| **News_Tone** | `newsemot` (News) | `Float32` | **Media Sentiment**. Daily average sentiment of financial news (1=Positive, 0=Neutral, -1=Negative). |
| **Newstone_t_30_t_3** | Derived | `Float32`<br>*Logic*: `shift(3).rolling(20).mean()` | **Historical Media Tone**. The average media sentiment over the past $[t-30, t-3]$ calendar days (approx. 20 trading days). Used as a strict control variable. |
| **Newsdummy_t_30_t_3** | Derived | `Int8` (Dummy)<br>*Logic*: `shift(3).rolling(20).max()` | **Historical Media Coverage**. 1 if the firm had any news coverage in the past $[t-30, t-3]$ window. |

### 3.1 Q&A

In [4]:
# ==============================================================================
# 3. 接入高频文本变量：互动问答 (Q&A) 与 财经新闻 (Financial News)
# ==============================================================================
print("========== 💬 Step 3: Merging Q&A and Financial News Signals ==========")

qna_panel_path = PATHS['processed'] / 'QnA_Slim_Panel.parquet'
if not qna_panel_path.exists():
    qna_panel_path = PATHS['processed'] / 'QnA_Panel.parquet'

if qna_panel_path.exists():
    print(f" -> 1. 加载互动面板: {qna_panel_path.name}")
    df_qna = pd.read_parquet(qna_panel_path)
    
    # 基础清洗与对齐
    df_qna['Stkcd'] = df_qna['Stkcd'].astype(str).str.zfill(6)
    df_qna['Date'] = pd.to_datetime(df_qna['Date'])
    df_qna = df_qna.drop_duplicates(subset=['Stkcd', 'Date'])
    df_qna = df_qna.drop(columns=['year', 'month'], errors='ignore')
    
    # =====================================================================
    # 显式选择需要保留的 QnA 变量 (剔除多余变量，精简特征空间)
    # =====================================================================
    columns_to_keep = [
        'Stkcd',               # 股票代码 (合并用主键，不可少)
        'Date',                # 日期 (合并用主键)
        'Num_Questions',       # 提问数量
        'TotalResponses',      # 回复数量
        'Reply_Rate',          # 回复率
        'Wrd_Count_Mean',      # 平均字数
        'Reply_Length_Mean',   # 平均回复长度
        'ResponseLag_Mean',    # 平均回复延迟
        'Investor_Tone',       # 投资者情感
        'Manager_Tone',        # 管理层情感
        'Sentiment_Gap',       # 情感差异 (Manager - Investor)
        'Substantiveness_ML',  # 回复实质性 (机器学习判定)
        'Aggregate_Tone'       # 综合情感 (Manager + Investor)
    ]

    # 安全过滤：取交集，确保只保留在 df_qna 中实际存在的列，防止 KeyError 报错
    actual_cols_to_keep = [col for col in columns_to_keep if col in df_qna.columns]
    df_qna = df_qna[actual_cols_to_keep]
    print(f"    ✅ 已重组 df_qna 变量，当前保留了 {len(df_qna.columns)} 列特征。")

    # =====================================================================
    # 【防重入逻辑】：如果 df_spine 已有这些特征列，则先删除再合并（防止生成 _x, _y）
    # =====================================================================
    # 提取除主键外的特征列名
    qna_features = [col for col in actual_cols_to_keep if col not in ['Stkcd', 'Date']]
    # 检查主表中是否已经存在这些特征
    existing_qna_cols = [col for col in qna_features if col in df_spine.columns]
    
    if existing_qna_cols:
        df_spine = df_spine.drop(columns=existing_qna_cols)
        print(f"    - [覆盖提示] 检测到主表已存在 {len(existing_qna_cols)} 个旧的 QnA 列，已自动清除以支持无缝覆盖。")
        
    # =====================================================================
    # 执行安全合并
    # =====================================================================
    df_spine = safe_merge(df_spine, df_qna, on_keys=['Stkcd', 'Date'])
    
else:
    print(f"❌ 未找到 QnA 面板：{qna_panel_path}")

========== 💬 Step 3: Merging Q&A and Financial News Signals ==========
 -> 1. 加载互动面板: QnA_Panel.parquet
    ✅ 已重组 df_qna 变量，当前保留了 13 列特征。


In [6]:
df_spine.columns

Index(['Stkcd', 'Date', 'Dretwd', 'Dretnd', 'Dsmvtll', 'OpenPrice',
       'ClosePrice', 'StateCode', 'LimitStatus', 'Rf_Daily', 'Rm_Daily',
       'Num_Questions', 'TotalResponses', 'Reply_Rate', 'Wrd_Count_Mean',
       'Reply_Length_Mean', 'ResponseLag_Mean', 'Investor_Tone',
       'Manager_Tone', 'Sentiment_Gap', 'Substantiveness_ML',
       'Aggregate_Tone'],
      dtype='object')

In [8]:
df_spine[['Num_Questions', 'Investor_Tone', 'Manager_Tone', 'Sentiment_Gap', 
          'Aggregate_Tone', 'Substantiveness_ML', 'Rf_Daily']].describe()

,Num_Questions,Investor_Tone,Manager_Tone,Sentiment_Gap,Aggregate_Tone,Substantiveness_ML,Rf_Daily
count,1.238632e+07,2.554308e+06,2.554308e+06,2.554308e+06,2.554308e+06,2.554308e+06,1.238632e+07
mean,4.003860e-01,-1.577127e-01,1.690537e-01,3.240069e-01,1.138516e-02,5.695351e-01,1.409858e-07
std,1.254191e+00,3.761273e-01,2.693382e-01,4.436530e-01,4.779753e-01,2.583752e-01,4.979608e-08
min,0.000000e+00,-9.313000e-01,-4.623000e-01,-1.003100e+00,-1.393600e+00,0.000000e+00,1.027778e-07
25%,0.000000e+00,-3.457000e-01,3.620816e-03,1.760000e-02,-2.524000e-01,3.961548e-01,1.138889e-07
50%,0.000000e+00,-6.500000e-03,1.011882e-01,2.719000e-01,1.190000e-02,6.211604e-01,1.138889e-07
75%,0.000000e+00,-1.800000e-03,3.084920e-01,6.592500e-01,2.977000e-01,7.842699e-01,1.500000e-07
max,2.470000e+02,9.038000e-01,9.212000e-01,1.833800e+00,1.825000e+00,9.435000e-01,2.611111e-07


### 3.2 Financial News

In [9]:

print(" -> 2. 加载网络财经新闻 (Financial News)...")

# 1. 穿透查找所有财经新闻的 Parquet 文件
news_files = list(PATHS['raw'].rglob("*网络财经新闻*.parquet"))

if not news_files:
    print("    ⚠️ 未找到网络财经新闻数据，请检查路径或文件名。")
else:
    df_news_list = []
    # 根据您的数据字典，原始文件中的真实列名
    raw_cols = ['Scode', 'Reptime', 'Newsemot']
    
    for f in tqdm(news_files, desc="Loading News Parquets"):
        try:
            # 仅读取需要的 3 列，极大地节省内存
            temp_df = pd.read_parquet(f, columns=raw_cols)
            
            # 重命名为标准列名
            temp_df = temp_df.rename(columns={
                'Scode': 'Stkcd', 
                'Reptime': 'Date', 
                'Newsemot': 'News_Tone'
            })
            
            # 🚨 清洗 Stkcd：转为 6 位字符串
            # (因为 df_spine 在 Step 2 结尾已经把 Stkcd 转回 6 位字符串了，这里必须对齐)
            temp_df['Stkcd'] = pd.to_numeric(temp_df['Stkcd'], errors='coerce')
            temp_df = temp_df.dropna(subset=['Stkcd'])
            temp_df['Stkcd'] = temp_df['Stkcd'].astype(int).astype(str).str.zfill(6)
            
            # 清洗 Date：转为 datetime 并截断到“日” (去掉时分秒)
            temp_df['Date'] = pd.to_datetime(temp_df['Date'], errors='coerce').dt.floor('D')
            
            # 清洗 News_Tone：转为浮点数
            temp_df['News_Tone'] = pd.to_numeric(temp_df['News_Tone'], errors='coerce')
            
            # 剔除空值
            temp_df = temp_df.dropna(subset=['Stkcd', 'Date', 'News_Tone'])
            df_news_list.append(temp_df)
            
        except Exception as e:
            print(f"    [报错] 读取 {f.name} 时出错: {e}")
            
    if df_news_list:
        df_news = pd.concat(df_news_list, ignore_index=True)
        del df_news_list
        gc.collect()
        
        # 2. 聚合到日度 (同一天同一只股票有多条新闻时，取平均情绪)
        print("    -> 聚合日度新闻情绪...")
        daily_news = df_news.groupby(['Stkcd', 'Date'])['News_Tone'].mean().reset_index()
        daily_news['News_Dummy'] = 1
        
        # 内存降维
        daily_news['News_Tone'] = daily_news['News_Tone'].astype('float32')
        daily_news['News_Dummy'] = daily_news['News_Dummy'].astype('int8')
        
        del df_news
        gc.collect()
        
        # 3. 🛡️ 安全合并到主干表
        print("    -> 合并至主干表...")
        df_spine = safe_merge(df_spine, daily_news, on_keys=['Stkcd', 'Date'])
        
        # 缺失值填 0 (没有新闻的日子，Dummy为0，Tone为0)
        df_spine['News_Dummy'] = df_spine['News_Dummy'].fillna(0).astype('int8')
        df_spine['News_Tone'] = df_spine['News_Tone'].fillna(0).astype('float32')
        
        del daily_news
        gc.collect()
        
        # 4. 🌟 计算 [t-30, t-3] 的滚动控制变量
        print("    -> 计算新闻控制变量 (Newstone_t_30_t_3, Newsdummy_t_30_t_3)...")
        # 必须先排序，保证 rolling 的时间序列正确
        df_spine.sort_values(['Stkcd', 'Date'], inplace=True)
        
        # 20 个交易日约等于 30 个自然日。shift(3) 保证了严格避开 t-2 到 t 的窗口
        df_spine['Newstone_t_30_t_3'] = df_spine.groupby('Stkcd')['News_Tone'].transform(
            lambda x: x.shift(3).rolling(20, min_periods=1).mean()
        ).fillna(0).astype('float32')
        
        df_spine['Newsdummy_t_30_t_3'] = df_spine.groupby('Stkcd')['News_Dummy'].transform(
            lambda x: x.shift(3).rolling(20, min_periods=1).max()
        ).fillna(0).astype('int8')
        
        # 清理原始的日度新闻列 (因为回归中我们只用滞后项作为控制变量)
        df_spine.drop(columns=['News_Tone', 'News_Dummy'], inplace=True)
        print("    ✅ 财经新闻特征接入完成！")

print("\n🚀 互动与新闻变量接入完成！")
print(f"当前全表面板维度: {df_spine.shape}")

# 预览检查
if 'Newstone_t_30_t_3' in df_spine.columns:
    print("\n👀 数据预览 (展示包含新闻特征的记录):")
    display(df_spine[['Stkcd', 'Date', 'Newstone_t_30_t_3', 'Newsdummy_t_30_t_3']].sample(5, random_state=42))

 -> 2. 加载网络财经新闻 (Financial News)...


Loading News Parquets:   0%|          | 0/37 [00:00<?, ?it/s]

    -> 聚合日度新闻情绪...
    -> 合并至主干表...
    -> 计算新闻控制变量 (Newstone_t_30_t_3, Newsdummy_t_30_t_3)...
    ✅ 财经新闻特征接入完成！

🚀 互动与新闻变量接入完成！
当前全表面板维度: (12386319, 24)

👀 数据预览 (展示包含新闻特征的记录):


,Stkcd,Date,Newstone_t_30_t_3,Newsdummy_t_30_t_3
4767856,300428,2019-04-16,0.050000,1
9090692,300621,2020-11-06,-0.075000,1
9064001,300598,2024-03-22,0.383333,1
320224,000748,2013-12-13,0.070000,1
10207750,600517,2022-11-09,0.000000,1


In [10]:
df_spine.columns

Index(['Stkcd', 'Date', 'Dretwd', 'Dretnd', 'Dsmvtll', 'OpenPrice',
       'ClosePrice', 'StateCode', 'LimitStatus', 'Rf_Daily', 'Rm_Daily',
       'Num_Questions', 'TotalResponses', 'Reply_Rate', 'Wrd_Count_Mean',
       'Reply_Length_Mean', 'ResponseLag_Mean', 'Investor_Tone',
       'Manager_Tone', 'Sentiment_Gap', 'Substantiveness_ML', 'Aggregate_Tone',
       'Newstone_t_30_t_3', 'Newsdummy_t_30_t_3'],
      dtype='object')

## 4: Firm Fundamentals & Earnings Quality
**Design Intent**: To construct accounting-based control variables and earnings quality proxies.

**Anti-Look-Ahead Bias**: We strictly ensure that trading days are only matched with financial reports that have already been publicly disclosed. For standard financials, we assume a 4-month reporting lag if the exact announcement date is missing. For earnings surprises, we strictly use the exact announcement date (`Annodt`).

| Final Variable | CSMAR Original Field | Data Type & Code Logic | Description |
| :--- | :--- | :--- | :--- |
| **Size** | `A001000000` | `Float32`<br>*Logic*: `Ln(Assets)` | **Firm Size**. The natural logarithm of Total Assets. A standard control variable for the size effect. |
| **Lev** | `A001000000`<br>`A002000000` | `Float32`<br>*Logic*: `Liabilities / Assets` | **Financial Leverage**. The ratio of Total Liabilities to Total Assets. Measures financial risk. |
| **ROA** | `B002000000`<br>`A001000000` | `Float32`<br>*Logic*: `NetIncome / Assets` | **Return on Assets**. The ratio of Net Income to Total Assets. A proxy for firm profitability. |
| **BM** | `F101001A` | `Float32` | **Book-to-Market Ratio**. Book value of equity divided by market capitalization. |
| **TBQ** | `F100901A` | `Float32` | **Tobin's Q**. Market value of the firm divided by the replacement cost of its assets. |
| **SUE** | `SUE` | `Float32`<br>*Logic*: Filter `TestPeriod == 6` | **Standardized Unexpected Earnings**. Proxy for fundamental information shocks. Merged strictly using `Annodt`. |
| **AdjEPS** | `AdjEPS` | `Float32` | **Adjusted EPS**. Supplementary earnings metric from the SUE dataset. |
| **Abs_DA** | `DisAcc` | `Float32`<br>*Logic*: `abs(DisAcc)` | **Absolute Discretionary Accruals**. Modified Jones Model. A higher value indicates lower earnings quality (more earnings management). |

In [11]:
# ==============================================================================
# 4.  财务、基本面与盈余质量数据接入 (Anti-Look-Ahead Bias & Extreme Memory Optimization)
# ==============================================================================
import gc
import pandas as pd
import numpy as np

print("========== 🛡️ Step 4: Merging Fundamentals & Earnings Quality ==========")



# ------------------------------------------------------------------------------
print(" -> 4.0 [内存急救] 正在压缩 df_spine 内存...")
float_cols = df_spine.select_dtypes(include=['float64']).columns
if len(float_cols) > 0:
    df_spine[float_cols] = df_spine[float_cols].astype('float32')

df_spine['Stkcd'] = pd.to_numeric(df_spine['Stkcd'], errors='coerce').astype('int32')
gc.collect()

# ------------------------------------------------------------------------------
# 4.1 自适应加载财务原始报表 (包含 SUE 和 盈余管理)
# ------------------------------------------------------------------------------
print(" -> 4.1 加载财务原始报表...")

def safe_load_fin(keyword, col_mapping):
    """自适应列名加载器，忽略缺失列，自动转换大小写别名，强制匹配文件夹中文名"""
    all_files = list(PATHS['raw'].rglob("*.parquet"))
    files = [f for f in all_files if keyword in str(f)]

    if not files:
        print(f"      [警告] 未在任何路径中找到包含 '{keyword}' 的 Parquet 文件。")
        return pd.DataFrame()

    df_list = []
    for f in files:
        try:
            df = pd.read_parquet(f)
            df.columns = [str(c).lower() for c in df.columns]

            rename_dict = {}
            for target_col, possible_names in col_mapping.items():
                for name in possible_names:
                    if name.lower() in df.columns:
                        rename_dict[name.lower()] = target_col
                        break

            df = df[list(rename_dict.keys())].rename(columns=rename_dict)

            if 'Stkcd' in df.columns:
                df['Stkcd'] = pd.to_numeric(df['Stkcd'], errors='coerce')
                df = df.dropna(subset=['Stkcd'])
                df['Stkcd'] = df['Stkcd'].astype('int32')
                df_list.append(df)
        except Exception as e:
            print(f"      [警告] 读取 {f.name} 时出错: {e}")

    if not df_list: return pd.DataFrame()
    return pd.concat(df_list, ignore_index=True)

# 字典映射定义
map_bal = {'Stkcd': ['stkcd', 'symbol'], 'Accper': ['accper'], 'Annodt': ['annodt'], 'Typrep': ['typrep'], 'Assets': ['a001000000'], 'Liabilities': ['a002000000']}
map_inc = {'Stkcd': ['stkcd', 'symbol'], 'Accper': ['accper'], 'Typrep': ['typrep'], 'NetIncome': ['b002000000']}
map_val = {'Stkcd': ['stkcd', 'symbol'], 'Accper': ['accper'], 'TBQ': ['f100901a'], 'BM': ['f101001a']}
map_sue = {'Stkcd': ['stkcd', 'symbol'], 'EndDate': ['enddate'], 'Annodt': ['annodt'], 'TestPeriod': ['testperiod'], 'SUE': ['sue'], 'AdjEPS': ['adjeps']}
map_em  = {'Stkcd': ['stkcd', 'symbol'], 'EndDate': ['enddate'], 'DisAcc': ['disacc']}

# 执行加载
df_bal = safe_load_fin('资产负债', map_bal)
df_inc = safe_load_fin('利润', map_inc)
df_val = safe_load_fin('相对价值', map_val)
df_sue = safe_load_fin('未预期盈余', map_sue)
df_em  = safe_load_fin('盈余管理', map_em)

# ------------------------------------------------------------------------------
# 4.2 计算核心财务指标 (Size, Lev, ROA)
# ------------------------------------------------------------------------------
print(" -> 4.2 计算核心财务指标 (Size, Lev, ROA)...")
df_fund = pd.DataFrame()

if not df_bal.empty and not df_inc.empty:
    if 'Accper' in df_bal.columns: df_bal['Accper'] = pd.to_datetime(df_bal['Accper'], errors='coerce')
    if 'Accper' in df_inc.columns: df_inc['Accper'] = pd.to_datetime(df_inc['Accper'], errors='coerce')
    if 'Accper' in df_val.columns: df_val['Accper'] = pd.to_datetime(df_val['Accper'], errors='coerce')

    if 'Typrep' in df_bal.columns: df_bal = df_bal[df_bal['Typrep'] == 'A']
    if 'Typrep' in df_inc.columns: df_inc = df_inc[df_inc['Typrep'] == 'A']

    df_fund = pd.merge(df_bal, df_inc[['Stkcd', 'Accper', 'NetIncome']], on=['Stkcd', 'Accper'], how='inner')
    if not df_val.empty:
        df_fund = pd.merge(df_fund, df_val, on=['Stkcd', 'Accper'], how='left')

    df_fund['Size'] = np.log(df_fund['Assets'].replace(0, np.nan))
    df_fund['Lev'] = df_fund['Liabilities'] / df_fund['Assets'].replace(0, np.nan)
    df_fund['ROA'] = df_fund['NetIncome'] / df_fund['Assets'].replace(0, np.nan)

    # 防御前视偏差 (延后4个月或使用实际公告日)
    if 'Annodt' in df_fund.columns:
        df_fund['Annodt'] = pd.to_datetime(df_fund['Annodt'], errors='coerce')
        df_fund['Effective_Date'] = df_fund['Annodt'].fillna(df_fund['Accper'] + pd.DateOffset(months=4))
    else:
        df_fund['Effective_Date'] = df_fund['Accper'] + pd.DateOffset(months=4)

    df_fund = df_fund.dropna(subset=['Stkcd', 'Effective_Date']).drop_duplicates(subset=['Stkcd', 'Effective_Date'])
    df_fund = df_fund[['Stkcd', 'Effective_Date', 'Size', 'Lev', 'ROA', 'BM', 'TBQ']].sort_values('Effective_Date')

del df_bal, df_inc, df_val
gc.collect()

# ------------------------------------------------------------------------------
# 4.3 处理 SUE 与 盈余管理 (Abs_DA)
# ------------------------------------------------------------------------------
print(" -> 4.3 处理 SUE 与 盈余管理 (Abs_DA)...")

# 处理 SUE
if not df_sue.empty:
    df_sue['TestPeriod'] = pd.to_numeric(df_sue['TestPeriod'], errors='coerce')
    df_sue = df_sue[df_sue['TestPeriod'] == 6].copy() # 仅保留半年/年度
    df_sue['Annodt'] = pd.to_datetime(df_sue['Annodt'], errors='coerce')

    # SUE 严格使用实际公告日作为生效日
    df_sue['Effective_Date'] = df_sue['Annodt']
    df_sue = df_sue.dropna(subset=['Stkcd', 'Effective_Date', 'SUE']).drop_duplicates(subset=['Stkcd', 'Effective_Date'])
    df_sue = df_sue[['Stkcd', 'Effective_Date', 'SUE', 'AdjEPS']].sort_values('Effective_Date')

# 处理 盈余管理 (Abs_DA)
if not df_em.empty:
    df_em['EndDate'] = pd.to_datetime(df_em['EndDate'], errors='coerce')
    df_em['Abs_DA'] = pd.to_numeric(df_em['DisAcc'], errors='coerce').abs()

    # 年度数据，延后4个月生效
    df_em['Effective_Date'] = df_em['EndDate'] + pd.DateOffset(months=4)
    df_em = df_em.dropna(subset=['Stkcd', 'Effective_Date', 'Abs_DA']).drop_duplicates(subset=['Stkcd', 'Effective_Date'])
    df_em = df_em[['Stkcd', 'Effective_Date', 'Abs_DA']].sort_values('Effective_Date')

# ------------------------------------------------------------------------------
# 4.4 序列化执行 pd.merge_asof (防止不同频率数据的时间错位)
# ------------------------------------------------------------------------------
print(" -> 4.4 执行 pd.merge_asof 向下填充财务与盈余数据...")



# =====================================================================
# 这是我们准备要并入的所有财务特征
fin_features = ['Size', 'Lev', 'ROA', 'BM', 'TBQ', 'SUE', 'AdjEPS', 'Abs_DA']

# 找出主表中存在的原版列，以及带 _x, _y 后缀的残骸列
cols_to_drop = [c for c in df_spine.columns if c in fin_features or any(c == f"{f}_x" or c == f"{f}_y" for f in fin_features)]

if cols_to_drop:
    df_spine = df_spine.drop(columns=cols_to_drop)
    print(f"    - [自动清理] 已清除遗留的同名及带后缀的财务垃圾列，共 {len(cols_to_drop)} 个。")
# =====================================================================

df_spine = df_spine.sort_values('Date')

# 1. 注入基础财务指标
if not df_fund.empty:
    df_spine = pd.merge_asof(df_spine, df_fund, left_on='Date', right_on='Effective_Date', by='Stkcd', direction='backward')
    df_spine.drop(columns=['Effective_Date'], inplace=True, errors='ignore')

# 2. 注入 SUE
if not df_sue.empty:
    df_spine = pd.merge_asof(df_spine, df_sue, left_on='Date', right_on='Effective_Date', by='Stkcd', direction='backward')
    df_spine.drop(columns=['Effective_Date'], inplace=True, errors='ignore')

# 3. 注入 Abs_DA
if not df_em.empty:
    df_spine = pd.merge_asof(df_spine, df_em, left_on='Date', right_on='Effective_Date', by='Stkcd', direction='backward')
    df_spine.drop(columns=['Effective_Date'], inplace=True, errors='ignore')

del df_fund, df_sue, df_em
gc.collect()

# ------------------------------------------------------------------------------
# 4.5 收尾：复原格式与预览
# ------------------------------------------------------------------------------
df_spine['Stkcd'] = df_spine['Stkcd'].astype(str).str.zfill(6)

print("\n🚀 财务与盈余质量指标接入完成（零前视偏差）！")
print(f"当前主干维度: {df_spine.shape}")

# 预览检查
check_cols = ['Stkcd', 'Date', 'Size', 'ROA', 'SUE', 'Abs_DA']
check_cols = [c for c in check_cols if c in df_spine.columns]
valid_rows = df_spine.dropna(subset=[c for c in ['Size', 'SUE'] if c in df_spine.columns], how='all')

if not valid_rows.empty:
    # 筛选出 check_cols 中没有任何空值 (NA) 的行
    perfect_match_rows = valid_rows.dropna(subset=check_cols)
    
    if not perfect_match_rows.empty:
        print("\n👀 数据预览 (展示 check_cols 中完全没有 NA 的匹配行):")
        # 动态计算抽样数量，防止完美匹配的行数少于 5 行时报错
        sample_size = min(5, len(perfect_match_rows))
        display(perfect_match_rows[check_cols].sample(sample_size, random_state=42))
    else:
        print("\n👀 [提示] 数据合并成功，但在这些行中 check_cols 至少都包含一个 NA。")
        # （可选）如果没有完美匹配的行，退回展示普通的合并结果
        sample_size = min(5, len(valid_rows))
        display(valid_rows[check_cols].sample(sample_size, random_state=42))
else:
    print("    [提示] 匹配失败，请检查原始数据。")

========== 🛡️ Step 4: Merging Fundamentals & Earnings Quality ==========
 -> 4.0 [内存急救] 正在压缩 df_spine 内存...
 -> 4.1 加载财务原始报表...
 -> 4.2 计算核心财务指标 (Size, Lev, ROA)...
 -> 4.3 处理 SUE 与 盈余管理 (Abs_DA)...
 -> 4.4 执行 pd.merge_asof 向下填充财务与盈余数据...

🚀 财务与盈余质量指标接入完成（零前视偏差）！
当前主干维度: (12386319, 32)

👀 数据预览 (展示 check_cols 中完全没有 NA 的匹配行):


,Stkcd,Date,Size,ROA,SUE,Abs_DA
2603116,000812,2014-09-23,21.194426,0.016603,0.338208,0.0174
2479847,600071,2014-07-10,20.653209,-0.044976,-2.120043,0.1111
7337371,600367,2020-11-05,21.420395,-0.004810,-1.869284,0.0213
5666178,000733,2019-01-17,22.959860,0.017640,0.261902,0.1212
4381375,600640,2017-07-03,22.576050,0.040999,-0.169742,0.0578


In [12]:
df_spine.columns

Index(['Stkcd', 'Date', 'Dretwd', 'Dretnd', 'Dsmvtll', 'OpenPrice',
       'ClosePrice', 'StateCode', 'LimitStatus', 'Rf_Daily', 'Rm_Daily',
       'Num_Questions', 'TotalResponses', 'Reply_Rate', 'Wrd_Count_Mean',
       'Reply_Length_Mean', 'ResponseLag_Mean', 'Investor_Tone',
       'Manager_Tone', 'Sentiment_Gap', 'Substantiveness_ML', 'Aggregate_Tone',
       'Newstone_t_30_t_3', 'Newsdummy_t_30_t_3', 'Size', 'Lev', 'ROA', 'BM',
       'TBQ', 'SUE', 'AdjEPS', 'Abs_DA'],
      dtype='object')

## 5: Corporate Governance, Attention & Ex-Post Actions
**Design Intent**: Incorporates annual/quarterly governance characteristics, analyst coverage, and ex-post corporate actions (insider trading & repurchases) into the daily spine.

**Anti-Look-Ahead Bias vs. Ex-Post Conditioning**:
- Historical features (`SOE`, `InsInvestorProp`, `AnaAttention`) are strictly lagged by 4 months to ensure public availability.
- Ex-post actions (`Insider_Sell_Next30d`, `Repurchase_Next3M`) are explicitly constructed as **forward-looking dummy variables**. They are *not* used as ex-ante predictors, but rather as ex-post conditioning filters for mechanism stress tests.

| Final Variable | CSMAR Original Field | Data Type & Code Logic | Description |
| :--- | :--- | :--- | :--- |
| **SOE** | `EquityNature` | `Float32` (Dummy) | **State-Owned Enterprise**. 1 if state-owned, 0 otherwise. |
| **TOP1** | `LargestHolderRate` | `Float32` | **Top 1 Ownership**. Percentage of shares held by the largest shareholder. |
| **DUAL** | `ConcurrentPosition` | `Float32` (Dummy) | **CEO Duality**. 1 if CEO and Chairman are the same person. |
| **BoardSize_Ln** | `BoardSize` | `Float32` | **Board Size**. Logarithm of the total number of directors. |
| **INDEP** | `IndDirectorRatio` | `Float32` | **Independent Director Ratio**. Percentage of independent directors. |
| **BALANCE** | `EquityBalance` | `Float32` | **Equity Check-and-Balance**. |
| **Firm_Age_Ln** | `Listdt` | `Float32` | **Log Firm Age**. Logarithm of years since IPO. |
| **InsInvestorProp**| `InsInvestorProp` | `Float32` | **Institutional Ownership**. Proportion of shares held by institutions. |
| **AnaAttention** | `AnaAttention` | `Float32` | **Analyst Coverage**. Number of analyst teams covering the firm. |
| **Insider_Sell_Next30d**| `ChangeDate`, `Shares` | `Int32` (Dummy) | **Ex-Post Insider Selling**. 1 if insiders net-sold shares within the *next 21 trading days* (approx. 30 calendar days). |
| **Repurchase_Next3M**| `DeclareDate` | `Int32` (Dummy) | **Ex-Post Share Repurchase**. 1 if the firm declared a share repurchase within the *next 63 trading days* (approx. 3 months). |

In [13]:
# ==============================================================================
# 5. 🛡️ 公司治理、关注度与事后行为接入 (Governance, Attention & Ex-Post Actions)
# ==============================================================================
import gc
import pandas as pd
import numpy as np

print("========== 🛡️ Step 5: Merging Governance, Attention & Ex-Post Actions ==========")

step5_cache_path = PATHS['processed'] / '05_df_spine_with_gov.parquet'


# ==============================================================================
FORCE_REBUILD_STEP5 = True

if step5_cache_path.exists() and not FORCE_REBUILD_STEP5:
    print(f" -> ⚡ 检测到本地缓存，直接加载: {step5_cache_path.name}")
    df_spine = pd.read_parquet(step5_cache_path)
    print(f"    ✅ 加载完成！当前主干维度: {df_spine.shape}")

else:
    if FORCE_REBUILD_STEP5 and step5_cache_path.exists():
        print(" -> 🔄 [调试模式] 已开启强制重跑，忽略并准备覆盖旧缓存文件...")
    else:
        print(" -> 未检测到本地缓存，开始执行合并与计算...")
        
    df_spine['Stkcd'] = pd.to_numeric(df_spine['Stkcd'], errors='coerce').astype('int32')

    # --------------------------------------------------------------------------
    # 5.1 自适应加载数据
    # --------------------------------------------------------------------------
    def load_gov_data(keyword, col_mapping):
        all_files = list(PATHS['raw'].rglob("*.parquet"))
        files = [f for f in all_files if keyword in str(f)]
        if not files: return pd.DataFrame()

        df_list = []
        for f in files:
            try:
                df = pd.read_parquet(f)
                df.columns = [str(c).lower() for c in df.columns]
                rename_dict = {name.lower(): target_col for target_col, possible_names in col_mapping.items()
                               for name in possible_names if name.lower() in df.columns}
                df = df[list(rename_dict.keys())].rename(columns=rename_dict)

                if 'Stkcd' in df.columns:
                    df['Stkcd'] = pd.to_numeric(df['Stkcd'], errors='coerce').astype('int32')
                df_list.append(df)
            except Exception: pass
        return pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()

    print(" -> 5.1 加载各模块数据...")
    df_equity  = load_gov_data('股权性质', {'Stkcd': ['stkcd', 'symbol'], 'EndDate': ['enddate'], 'EquityNature': ['equitynature'], 'TOP1': ['largestholderrate']})
    df_board   = load_gov_data('治理能力', {'Stkcd': ['stkcd', 'symbol'], 'EndDate': ['enddate'], 'DUAL': ['concurrentposition'], 'BoardSize': ['boardsize'], 'INDEP': ['inddirectorratio']})
    df_balance = load_gov_data('混改', {'Stkcd': ['stkcd', 'symbol'], 'EndDate': ['enddate'], 'BALANCE': ['equitybalance']})
    df_ipo     = load_gov_data('上市日期', {'Stkcd': ['stkcd', 'symbol'], 'Listdt': ['listdt', 'listeddate']})

    # 新增变量加载
    df_inst    = load_gov_data('机构持股', {'Stkcd': ['stkcd', 'symbol'], 'EndDate': ['enddate'], 'InsInvestorProp': ['insinvestorprop']})
    df_ana     = load_gov_data('分析师', {'Stkcd': ['stkcd', 'symbol'], 'Accper': ['accper'], 'AnaAttention': ['anaattention']})
    df_insider = load_gov_data('董监高', {'Stkcd': ['stkcd', 'symbol'], 'ChangeDate': ['changedate'], 'AfterShares': ['afterchangingshares'], 'BeforeShares': ['beforechangingshares']})
    df_rep     = load_gov_data('回购', {'Stkcd': ['stkcd', 'symbol'], 'DeclareDate': ['declaredate']})

    # --------------------------------------------------------------------------
    # 5.2 拼装历史特征 (需防前视偏差)
    # --------------------------------------------------------------------------
    print(" -> 5.2 拼装并处理历史特征 (防前视偏差)...")
    df_gov = pd.DataFrame()

    # 基础治理变量
    if not df_equity.empty:
        df_equity = df_equity[pd.to_datetime(df_equity['EndDate'], errors='coerce').dt.month == 12].copy()
        df_equity['SOE'] = np.where(df_equity['EquityNature'].astype(str).str.contains('国企|国有'), 1, 0)
        df_equity['TOP1'] = pd.to_numeric(df_equity['TOP1'], errors='coerce') / 100.0
        df_gov = df_equity[['Stkcd', 'EndDate', 'SOE', 'TOP1']]

    if not df_board.empty:
        df_board = df_board[pd.to_datetime(df_board['EndDate'], errors='coerce').dt.month == 12].copy()
        df_board['DUAL'] = pd.to_numeric(df_board['DUAL'], errors='coerce')
        df_board['BoardSize_Ln'] = np.log(pd.to_numeric(df_board['BoardSize'], errors='coerce').replace(0, np.nan))
        df_board['INDEP'] = pd.to_numeric(df_board['INDEP'], errors='coerce')
        df_gov = pd.merge(df_gov, df_board[['Stkcd', 'EndDate', 'DUAL', 'BoardSize_Ln', 'INDEP']], on=['Stkcd', 'EndDate'], how='outer') if not df_gov.empty else df_board

    if not df_balance.empty:
        df_balance = df_balance[pd.to_datetime(df_balance['EndDate'], errors='coerce').dt.month == 12].copy()
        df_balance['BALANCE'] = pd.to_numeric(df_balance['BALANCE'], errors='coerce')
        df_gov = pd.merge(df_gov, df_balance[['Stkcd', 'EndDate', 'BALANCE']], on=['Stkcd', 'EndDate'], how='outer') if not df_gov.empty else df_balance

    # 机构持股 (季度/半年度)
    if not df_inst.empty:
        df_inst['InsInvestorProp'] = pd.to_numeric(df_inst['InsInvestorProp'], errors='coerce')
        df_gov = pd.merge(df_gov, df_inst[['Stkcd', 'EndDate', 'InsInvestorProp']], on=['Stkcd', 'EndDate'], how='outer') if not df_gov.empty else df_inst

    # 分析师关注度 (年度)
    if not df_ana.empty:
        df_ana['EndDate'] = df_ana['Accper'] # 统一列名以便合并
        df_ana['AnaAttention'] = pd.to_numeric(df_ana['AnaAttention'], errors='coerce')
        df_gov = pd.merge(df_gov, df_ana[['Stkcd', 'EndDate', 'AnaAttention']], on=['Stkcd', 'EndDate'], how='outer') if not df_gov.empty else df_ana

    # 统一延后 4 个月生效
    if not df_gov.empty:
        df_gov['EndDate'] = pd.to_datetime(df_gov['EndDate'], errors='coerce')
        df_gov['Effective_Date'] = df_gov['EndDate'] + pd.DateOffset(months=4)
        df_gov = df_gov.dropna(subset=['Stkcd', 'Effective_Date']).drop_duplicates(subset=['Stkcd', 'Effective_Date'])
        df_gov = df_gov.sort_values('Effective_Date').drop(columns=['EndDate'])

        # 注入主干表
        df_spine = df_spine.sort_values('Date')
        df_spine = pd.merge_asof(df_spine, df_gov, left_on='Date', right_on='Effective_Date', by='Stkcd', direction='backward')
        df_spine.drop(columns=['Effective_Date'], inplace=True, errors='ignore')

    del df_gov, df_equity, df_board, df_balance, df_inst, df_ana
    gc.collect()

    # --------------------------------------------------------------------------
    # 5.3 构造事后条件变量 (Ex-Post Actions: Insider & Repurchase)
    # --------------------------------------------------------------------------
    print(" -> 5.3 构造事后条件变量 (Insider Selling & Repurchases)...")

    # 内部人减持 (Insider Selling)
    if not df_insider.empty:
        df_insider['Date'] = pd.to_datetime(df_insider['ChangeDate'], errors='coerce')
        df_insider['Net_Change'] = pd.to_numeric(df_insider['AfterShares'], errors='coerce').fillna(0) - pd.to_numeric(df_insider['BeforeShares'], errors='coerce').fillna(0)
        daily_insider = df_insider.groupby(['Stkcd', 'Date'])['Net_Change'].sum().reset_index()

        # 精确匹配到交易日历
        df_spine = pd.merge(df_spine, daily_insider, on=['Stkcd', 'Date'], how='left')
        df_spine['Net_Change'] = df_spine['Net_Change'].fillna(0)

        # 前向滚动 21 个交易日 (约 30 个自然日) 寻找是否有减持
        df_spine.sort_values(['Stkcd', 'Date'], inplace=True)
        df_spine['Min_Change_Next30d'] = df_spine.groupby('Stkcd')['Net_Change'].transform(lambda x: x.shift(-1).rolling(21, min_periods=1).min())
        df_spine['Insider_Sell_Next30d'] = (df_spine['Min_Change_Next30d'] < 0).astype('int8')
        df_spine.drop(columns=['Net_Change', 'Min_Change_Next30d'], inplace=True)

    # 股份回购 (Repurchases)
    if not df_rep.empty:
        df_rep['Date'] = pd.to_datetime(df_rep['DeclareDate'], errors='coerce')
        df_rep['Has_Repurchase'] = 1
        daily_rep = df_rep.groupby(['Stkcd', 'Date'])['Has_Repurchase'].max().reset_index()

        # 精确匹配到交易日历
        df_spine = pd.merge(df_spine, daily_rep, on=['Stkcd', 'Date'], how='left')
        df_spine['Has_Repurchase'] = df_spine['Has_Repurchase'].fillna(0)

        # 前向滚动 63 个交易日 (约 3 个月) 寻找是否有回购
        df_spine.sort_values(['Stkcd', 'Date'], inplace=True)
        df_spine['Repurchase_Next3M'] = df_spine.groupby('Stkcd')['Has_Repurchase'].transform(lambda x: x.shift(-1).rolling(63, min_periods=1).max()).fillna(0).astype('int8')
        df_spine.drop(columns=['Has_Repurchase'], inplace=True)

    del df_insider, df_rep
    gc.collect()

    # --------------------------------------------------------------------------
    # 5.4 接入上市年限 (Firm Age)
    # --------------------------------------------------------------------------
    if not df_ipo.empty:
        df_ipo['Listdt'] = pd.to_datetime(df_ipo['Listdt'], errors='coerce')
        df_ipo = df_ipo.dropna(subset=['Stkcd', 'Listdt']).drop_duplicates(subset=['Stkcd'])
        df_spine = pd.merge(df_spine, df_ipo[['Stkcd', 'Listdt']], on='Stkcd', how='left')
        df_spine['Firm_Age'] = (df_spine['Date'] - df_spine['Listdt']).dt.days / 365.25
        df_spine.loc[df_spine['Firm_Age'] <= 0, 'Firm_Age'] = np.nan
        df_spine['Firm_Age_Ln'] = np.log(df_spine['Firm_Age']).astype('float32')
        df_spine.drop(columns=['Listdt', 'Firm_Age'], inplace=True, errors='ignore')
    del df_ipo
    gc.collect()

    # --------------------------------------------------------------------------
    # 5.5 收尾与保存 Checkpoint
    # --------------------------------------------------------------------------
    print(" -> 5.5 还原格式并保存本地缓存...")
    df_spine['Stkcd'] = df_spine['Stkcd'].astype(str).str.zfill(6)

    df_spine.to_parquet(step5_cache_path, index=False)
    print(f"    💾 数据已成功强制覆盖缓存至: {step5_cache_path}")

    print("\n🚀 治理、关注度与事后特征接入完成！")
    print(f"当前主干维度: {df_spine.shape}")

# ==============================================================================
# 安全预览机制 (升级版：优先展示完全无缺失值的行)
# ==============================================================================
check_cols = ['Stkcd', 'Date', 'SOE', 'InsInvestorProp', 'AnaAttention', 'Insider_Sell_Next30d', 'Repurchase_Next3M']
check_cols = [c for c in check_cols if c in df_spine.columns]

if len(check_cols) > 2:
    valid_rows = df_spine.dropna(subset=[c for c in ['InsInvestorProp', 'AnaAttention'] if c in df_spine.columns], how='all')
    
    if not valid_rows.empty:
        perfect_match_rows = valid_rows.dropna(subset=check_cols)
        
        if not perfect_match_rows.empty:
            print("\n👀 数据预览 (展示 check_cols 中完全没有 NA 的匹配行):")
            sample_size = min(5, len(perfect_match_rows))
            display(perfect_match_rows[check_cols].sample(sample_size, random_state=42))
        else:
            print("\n👀 [提示] 数据合并成功，但在这些行中 check_cols 至少都包含一个 NA。")
            sample_size = min(5, len(valid_rows))
            display(valid_rows[check_cols].sample(sample_size, random_state=42))

========== 🛡️ Step 5: Merging Governance, Attention & Ex-Post Actions ==========
 -> 🔄 [调试模式] 已开启强制重跑，忽略并准备覆盖旧缓存文件...
 -> 5.1 加载各模块数据...
 -> 5.2 拼装并处理历史特征 (防前视偏差)...
 -> 5.3 构造事后条件变量 (Insider Selling & Repurchases)...
 -> 5.5 还原格式并保存本地缓存...
    💾 数据已成功强制覆盖缓存至: D:\iip_asset_pricing\data\processed\05_df_spine_with_gov.parquet

🚀 治理、关注度与事后特征接入完成！
当前主干维度: (12386319, 43)

👀 数据预览 (展示 check_cols 中完全没有 NA 的匹配行):


,Stkcd,Date,SOE,InsInvestorProp,AnaAttention,Insider_Sell_Next30d,Repurchase_Next3M
10236557,601669,2020-05-27,1.0,77.2373,8.0,0,0
10468792,603008,2023-06-30,0.0,51.9731,34.0,0,0
10933425,603421,2017-06-16,0.0,0.1325,1.0,0,0
5972533,300432,2018-06-12,0.0,34.1216,14.0,0,0
1284181,000892,2019-06-04,0.0,51.8966,4.0,0,0


In [14]:
df_spine.columns

Index(['Stkcd', 'Date', 'Dretwd', 'Dretnd', 'Dsmvtll', 'OpenPrice',
       'ClosePrice', 'StateCode', 'LimitStatus', 'Rf_Daily', 'Rm_Daily',
       'Num_Questions', 'TotalResponses', 'Reply_Rate', 'Wrd_Count_Mean',
       'Reply_Length_Mean', 'ResponseLag_Mean', 'Investor_Tone',
       'Manager_Tone', 'Sentiment_Gap', 'Substantiveness_ML', 'Aggregate_Tone',
       'Newstone_t_30_t_3', 'Newsdummy_t_30_t_3', 'Size', 'Lev', 'ROA', 'BM',
       'TBQ', 'SUE', 'AdjEPS', 'Abs_DA', 'SOE', 'TOP1', 'DUAL', 'BoardSize_Ln',
       'INDEP', 'BALANCE', 'InsInvestorProp', 'AnaAttention',
       'Insider_Sell_Next30d', 'Repurchase_Next3M', 'Firm_Age_Ln'],
      dtype='object')

## 6: Market Microstructure, Pricing Factors & Macro Index
**Design Intent**: Integrates high-frequency microstructure proxies, standard asset pricing factors, and macro sentiment indices.

| Final Variable | Original Field / Source | Description |
| :--- | :--- | :--- |
| **Imb_Noise** | `Imb_Noise` (Tan 2024) | **Retail Order Imbalance**. Net buying pressure from retail investors. |
| **Imb_Smart** | `Imb_Smart` (Tan 2024) | **Institutional Order Imbalance**. Net buying pressure from institutional investors. |
| **Ln_MarginBuy** | `Financeamount` | **Margin Trading**. Logarithm of daily margin buying amount. |
| **Dturn** | `Dturn` | **Monthly Turnover Rate**. Proxy for stock liquidity and investor attention. |
| **Beta** | `Beta` | **Market Beta**. Systemic risk coefficient. |
| **Illiqd** | `Illiqd` | **Amihud Illiquidity Ratio**. Absolute return divided by trading volume. |
| **CH4 / q5 / FF3** | Factor Datasets | **Asset Pricing Factors**. Used later for calculating Abnormal Returns (ARet). |
| **Sentiment_Index**| `Sentiment_Index` | **Baker & Wurgler (2006) Index**. Aggregate market-level retail investor sentiment in China. |



In [15]:
# ==============================================================================
# 6. Step 6: Merging Microstructure, Factors & Macro 
# ==============================================================================
import gc
import pandas as pd
import numpy as np

print("========== Step 6: Merging Microstructure, Factors & Macro ==========")

# ------------------------------------------------------------------------------
# 6.0 智能内存复用与全局防重入清理 
# ------------------------------------------------------------------------------
step5_cache_path = PATHS['processed'] / '05_df_spine_with_gov.parquet'

if 'df_spine' in locals() or 'df_spine' in globals():
    print(" -> ⚡ df_spine 已在内存中，直接复用。")
elif step5_cache_path.exists():
    print(f" -> ⚡ 内存为空，从 Step 5 缓存极速加载: {step5_cache_path.name}")
    df_spine = pd.read_parquet(step5_cache_path)
else:
    raise FileNotFoundError("找不到基础数据，请先确保跑通了 Step 5！")

df_spine['Stkcd'] = pd.to_numeric(df_spine['Stkcd'], errors='coerce').astype('int32')
if 'year' not in df_spine.columns:
    df_spine['year'] = df_spine['Date'].dt.year
    df_spine['month'] = df_spine['Date'].dt.month



# =====================================================================
step6_features = [
    'Imb_Noise', 'Imb_Smart', 'Ln_MarginBuy',              # 6.1 高频微观
    'Dturn', 'Beta', 'Illiqd',                             # 6.2 月度微观
    'ch4_mktrf', 'ch4_SMB', 'ch4_VMG', 'ch4_PMO',          # 6.3 CH4 因子
    'q5_MKT', 'q5_ME', 'q5_IA', 'q5_ROE', 'q5_EG',         # 6.3 q5 因子
    'Mkt_RF', 'SMB', 'HML',                                # 6.3 FF3 因子
    'Sentiment_Index'                                      # 6.4 宏观情绪
]

# 找出主表中已经存在的原版列，以及带 _x, _y 后缀的“历史遗留”列
cols_to_drop = [c for c in df_spine.columns if c in step6_features or any(c == f"{f}_x" or c == f"{f}_y" for f in step6_features)]

if cols_to_drop:
    print(f" -> 🧹 [防重入清理] 准备就地清除 {len(cols_to_drop)} 个旧特征，释放内存...")
    # 【极致内存优化】：放弃使用 .drop()，改用原生循环 del，拒绝内存翻倍！
    for col in cols_to_drop:
        del df_spine[col]
    
    # 强行召唤垃圾回收，立刻清空剪断的变量内存
    gc.collect()
    print("    ✅ 清理完成，内存已安全释放！")


float64_cols = df_spine.select_dtypes(include=['float64']).columns
if len(float64_cols) > 0:
    print(f" -> 🗜️ 正在将主表中 {len(float64_cols)} 个 float64 列压缩至 float32...")
    df_spine[float64_cols] = df_spine[float64_cols].astype('float32')
    gc.collect()
# =====================================================================


# --------------------------------------------------------------------------
# 6.1 加载日度微观结构 (Order Imbalance & Margin Trading)
# --------------------------------------------------------------------------
print(" -> 6.1 加载日度微观结构 (OIB & Margin Trading)...")

# 1. 订单不平衡 (Order Imbalance)
df_oib = load_optimized_parquet('Imbalance_Tan2024', usecols=['Stkcd', 'Date', 'Imb_Noise', 'Imb_Smart'])
if not df_oib.empty:
    df_spine = safe_merge(df_spine, df_oib, on_keys=['Stkcd', 'Date'])
    print("    ✅ 订单不平衡 (Imb_Noise, Imb_Smart) 接入成功！")
else:
    print("    ⚠️ 未找到订单不平衡数据 (Imbalance_Tan2024)。")

# 2. 融资融券 (Margin Trading)
map_margin = {'Stkcd': ['stockcode'], 'Date': ['mtdate'], 'Financeamount': ['financeamount']}
df_margin = load_optimized_parquet('融资融券', usecols=['Stockcode', 'Mtdate', 'Financeamount'], rename_dict={'Stockcode': 'Stkcd', 'Mtdate': 'Date'})
if not df_margin.empty:
    df_margin['Ln_MarginBuy'] = np.log1p(pd.to_numeric(df_margin['Financeamount'], errors='coerce').fillna(0)).astype('float32')
    df_spine = safe_merge(df_spine, df_margin[['Stkcd', 'Date', 'Ln_MarginBuy']], on_keys=['Stkcd', 'Date'])
    print("    ✅ 融资买入额 (Ln_MarginBuy) 接入成功！")
del df_oib, df_margin
gc.collect()


# --------------------------------------------------------------------------
# 6.2 加载个股月度微观特征 (Illiqd, Beta, Dturn) -> 纯数学 Hash 映射
# --------------------------------------------------------------------------
print(" -> 6.2 加载月度微观特征 (SRFR_Amnthlyr)...")
srfr_files = list(PATHS['raw'].rglob("*月个股回报*.parquet")) + list(PATHS['raw'].rglob("*SRFR*.parquet"))
if srfr_files:
    df_srfr_list = []
    for f in srfr_files:
        try:
            df_temp = pd.read_parquet(f)
            df_temp.columns = [str(c).lower() for c in df_temp.columns]
            use_cols = {col: (col.capitalize() if col not in ['stkcd', 'trdmnt'] else col)
                        for col in ['stkcd', 'trdmnt', 'dturn', 'beta', 'illiqd'] if col in df_temp.columns}

            if 'stkcd' in use_cols and 'trdmnt' in use_cols:
                df_temp = df_temp[list(use_cols.keys())].rename(columns=use_cols)
                df_temp['Stkcd'] = pd.to_numeric(df_temp['stkcd'], errors='coerce').astype('int32')
                df_temp['trdmnt'] = pd.to_datetime(df_temp['trdmnt'], errors='coerce')
                df_temp['year'] = df_temp['trdmnt'].dt.year
                df_temp['month'] = df_temp['trdmnt'].dt.month
                df_temp.dropna(subset=['Stkcd', 'year', 'month'], inplace=True)
                df_srfr_list.append(df_temp)
        except Exception: pass

    if df_srfr_list:
        df_micro = pd.concat(df_srfr_list, ignore_index=True).drop_duplicates(subset=['Stkcd', 'year', 'month'])

        # 构建 int64 Hash 避免内存爆炸
        hash_spine = df_spine['Stkcd'].astype('int64') * 1000000 + df_spine['year'].astype('int64') * 100 + df_spine['month'].astype('int64')
        hash_micro = df_micro['Stkcd'].astype('int64') * 1000000 + df_micro['year'].astype('int64') * 100 + df_micro['month'].astype('int64')
        df_micro.index = hash_micro

        for col in ['Dturn', 'Beta', 'Illiqd']:
            if col in df_micro.columns:
                df_spine[col] = hash_spine.map(df_micro[col]).astype('float32')
                print(f"        ✅ {col} 映射成功，有效匹配: {df_spine[col].notna().sum():,} 行")

        del df_micro, hash_spine, hash_micro
        gc.collect()


# --------------------------------------------------------------------------
# 6.3 加载资产定价因子 (CH4, q5, FF3) -> 纯数字 Hash 映射
# --------------------------------------------------------------------------
print(" -> 6.3 加载日度资产定价因子 (CH4, q5, FF3)...")
spine_date_hash = (df_spine['Date'].dt.year * 10000 + df_spine['Date'].dt.month * 100 + df_spine['Date'].dt.day).astype('int32')

def map_time_series_factor(keyword, date_col_possible, rename_dict=None):
    files = list(PATHS['raw'].rglob(f"*{keyword}*.parquet"))
    if not files: return
    try:
        df_fac = pd.read_parquet(files[0])
        date_col = next((c for c in df_fac.columns if str(c).lower() in [dc.lower() for dc in date_col_possible]), None)
        if not date_col: return

        market_col = next((c for c in df_fac.columns if str(c).lower() in ['markettypeid', 'markettype']), None)
        if market_col:
            available_types = df_fac[market_col].astype(str).unique()
            if 'P9714' in available_types: df_fac = df_fac[df_fac[market_col].astype(str) == 'P9714']
            elif 'P9709' in available_types: df_fac = df_fac[df_fac[market_col].astype(str) == 'P9709']
            elif 'P9706' in available_types: df_fac = df_fac[df_fac[market_col].astype(str) == 'P9706']

        if rename_dict:
            cols_to_keep = [date_col] + [c for c in rename_dict.keys() if c in df_fac.columns]
            df_fac = df_fac[cols_to_keep].rename(columns=rename_dict)

        # 【防 1970 纳秒漏洞】：强制将日期列转为字符型再解析
        parsed_dates = pd.to_datetime(df_fac[date_col].astype(str), errors='coerce')
        df_fac['date_hash'] = (parsed_dates.dt.year * 10000 +
                               parsed_dates.dt.month * 100 +
                               parsed_dates.dt.day).astype('int32')
        
        df_fac = df_fac.dropna(subset=['date_hash']).drop_duplicates(subset=['date_hash']).set_index('date_hash')

        for col in df_fac.columns:
            if col != date_col and col != 'date_hash':
                df_spine[col] = spine_date_hash.map(df_fac[col]).astype('float32')
        print(f"    ✅ {keyword} 因子映射完成。")
    except Exception as e: 
        print(f"    ❌ {keyword} 映射出错: {e}")

# 执行映射
map_time_series_factor('CH4', ['date', 'trddt', 'tradingdate'], {'mktrf': 'ch4_mktrf', 'SMB': 'ch4_SMB', 'VMG': 'ch4_VMG', 'PMO': 'ch4_PMO'})
map_time_series_factor('q5', ['date', 'trddt', 'DATE', 'tradingdate'], {'R_MKT': 'q5_MKT', 'R_ME': 'q5_ME', 'R_IA': 'q5_IA', 'R_ROE': 'q5_ROE', 'R_EG': 'q5_EG'})
map_time_series_factor('THRFACDAY', ['trddt', 'date', 'tradingdate'], {'RiskPremium1': 'Mkt_RF', 'SMB1': 'SMB', 'HML1': 'HML'})


# --------------------------------------------------------------------------
# 6.4 加载宏观情绪指数 (BW Index)
# --------------------------------------------------------------------------
print(" -> 6.4 加载中国宏观投资者情绪指数 (BW Index)...")
bw_files = list(PATHS['raw'].rglob("*BW_Sentiment*.parquet")) + list(PATHS['raw'].rglob("*BW_Sentiment*.csv"))
if bw_files:
    try:
        f_bw = bw_files[0]
        df_bw = pd.read_csv(f_bw) if f_bw.suffix == '.csv' else pd.read_parquet(f_bw)
        if 'Month' in df_bw.columns and 'Sentiment_Index' in df_bw.columns:
            # 【防 1970 纳秒漏洞】：强制转为字符
            df_bw['Month'] = pd.to_datetime(df_bw['Month'].astype(str), errors='coerce')
            hash_spine = (df_spine['year'] * 100 + df_spine['month']).astype('int32')
            hash_bw = (df_bw['Month'].dt.year * 100 + df_bw['Month'].dt.month).astype('int32')
            df_bw.index = hash_bw
            df_spine['Sentiment_Index'] = hash_spine.map(df_bw['Sentiment_Index']).astype('float32')
            print(f"    ✅ 宏观情绪指数映射完成，有效匹配: {df_spine['Sentiment_Index'].notna().sum():,} 行")
    except Exception as e: 
        print(f"    ❌ BW 情绪指数映射出错: {e}")

print("\n🚀 Step 6 执行完毕！所有微观特征及宏观因子均已安全并入 df_spine。")

========== 🛡️ Step 6: Merging Microstructure, Factors & Macro ==========
 -> ⚡ df_spine 已在内存中，直接复用。
 -> 🗜️ 正在将主表中 16 个 float64 列压缩至 float32...
 -> 6.1 加载日度微观结构 (OIB & Margin Trading)...
    ✅ 订单不平衡 (Imb_Noise, Imb_Smart) 接入成功！
    ✅ 融资买入额 (Ln_MarginBuy) 接入成功！
 -> 6.2 加载月度微观特征 (SRFR_Amnthlyr)...
        ✅ Dturn 映射成功，有效匹配: 12,386,284 行
        ✅ Beta 映射成功，有效匹配: 8,952,471 行
        ✅ Illiqd 映射成功，有效匹配: 12,386,284 行
 -> 6.3 加载日度资产定价因子 (CH4, q5, FF3)...
    ✅ CH4 因子映射完成。
    ✅ q5 因子映射完成。
    ✅ THRFACDAY 因子映射完成。
 -> 6.4 加载中国宏观投资者情绪指数 (BW Index)...
    ✅ 宏观情绪指数映射完成，有效匹配: 12,324,574 行

🚀 Step 6 执行完毕！所有微观特征及宏观因子均已安全并入 df_spine。


In [16]:
df_spine.columns

Index(['Stkcd', 'Date', 'Dretwd', 'Dretnd', 'Dsmvtll', 'OpenPrice',
       'ClosePrice', 'StateCode', 'LimitStatus', 'Rf_Daily', 'Rm_Daily',
       'Num_Questions', 'TotalResponses', 'Reply_Rate', 'Wrd_Count_Mean',
       'Reply_Length_Mean', 'ResponseLag_Mean', 'Investor_Tone',
       'Manager_Tone', 'Sentiment_Gap', 'Substantiveness_ML', 'Aggregate_Tone',
       'Newstone_t_30_t_3', 'Newsdummy_t_30_t_3', 'Size', 'Lev', 'ROA', 'BM',
       'TBQ', 'SUE', 'AdjEPS', 'Abs_DA', 'SOE', 'TOP1', 'DUAL', 'BoardSize_Ln',
       'INDEP', 'BALANCE', 'InsInvestorProp', 'AnaAttention',
       'Insider_Sell_Next30d', 'Repurchase_Next3M', 'Firm_Age_Ln', 'year',
       'month', 'Imb_Noise', 'Imb_Smart', 'Ln_MarginBuy', 'Dturn', 'Beta',
       'Illiqd', 'ch4_mktrf', 'ch4_SMB', 'ch4_VMG', 'ch4_PMO', 'q5_MKT',
       'q5_ME', 'q5_IA', 'q5_ROE', 'q5_EG', 'Mkt_RF', 'SMB', 'HML',
       'Sentiment_Index'],
      dtype='object')

In [22]:
df_spine['Newstone_t_30_t_3'].describe()

count    1.238632e+07
mean     4.924263e-02
std      1.193689e-01
min     -1.000000e+00
25%      0.000000e+00
50%      2.500000e-02
75%      1.000000e-01
max      1.000000e+00
Name: Newstone_t_30_t_3, dtype: float64

## 7. 基础面板最终清理与保存 

In [17]:
print("\n========== 💾 Step 7: Final Base Panel Export ==========")

# 1. 清理冗余列
print(" -> 正在清理冗余列...")
df_spine.drop(columns=['year', 'month'], inplace=True, errors='ignore')
if 'spine_date_hash' in locals(): del spine_date_hash
gc.collect()

# 2. 还原 Stkcd 格式为 6 位字符串
df_spine['Stkcd'] = df_spine['Stkcd'].astype(str).str.zfill(6)

# 3. 原地排序保证时间序列连续性
print(" -> 正在执行原地排序 (保证时间序列连续性)...")
df_spine.sort_values(['Stkcd', 'Date'], inplace=True)

# 4. 另存为 Advanced 版本，保护原数据
final_advanced_path = PATHS['processed'] / '01_Base_Daily_Panel_Advanced.parquet'
print(f" -> 正在将终极基础大表保存至: {final_advanced_path.name}")
df_spine.to_parquet(final_advanced_path, index=False)

print(f"\n🎉 恭喜！包含所有高级机制变量的本地化基础大表已生成！")
print(f" -> 总行数: {len(df_spine):,}")
print(f" -> 总列数: {len(df_spine.columns)}")

# 最终预览
print("\n👀 终极数据预览 (随机展示 5 行包含高级特征的数据):")
check_cols = ['Stkcd', 'Date', 'Dretwd', 'Imb_Noise', 'Ln_MarginBuy', 'ch4_PMO', 'Sentiment_Index']
check_cols = [c for c in check_cols if c in df_spine.columns]
if len(check_cols) > 3:
    display(df_spine.dropna(subset=[c for c in ['Imb_Noise', 'Ln_MarginBuy'] if c in df_spine.columns], how='any')[check_cols].sample(5, random_state=42))


========== 💾 Step 7: Final Base Panel Export ==========
 -> 正在清理冗余列...
 -> 正在执行原地排序 (保证时间序列连续性)...
 -> 正在将终极基础大表保存至: 01_Base_Daily_Panel_Advanced.parquet

🎉 恭喜！包含所有高级机制变量的本地化基础大表已生成！
 -> 总行数: 12,386,319
 -> 总列数: 62

👀 终极数据预览 (随机展示 5 行包含高级特征的数据):


,Stkcd,Date,Dretwd,Imb_Noise,Ln_MarginBuy,ch4_PMO,Sentiment_Index
316765,000425,2012-11-19,0.004878,-0.027534,14.756830,-0.0019,0.371140
1000024,000762,2012-01-31,0.000455,0.008671,15.142608,-0.0027,-1.915888
1489575,000970,2013-07-19,-0.072364,0.006636,17.774275,-0.0047,-1.409076
1206423,000858,2013-07-01,-0.003493,-0.000554,16.907175,-0.0142,-1.409076
1184989,000839,2012-09-04,-0.009677,-0.158062,14.057370,-0.0035,-1.726412


In [18]:
df_spine.columns

Index(['Stkcd', 'Date', 'Dretwd', 'Dretnd', 'Dsmvtll', 'OpenPrice',
       'ClosePrice', 'StateCode', 'LimitStatus', 'Rf_Daily', 'Rm_Daily',
       'Num_Questions', 'TotalResponses', 'Reply_Rate', 'Wrd_Count_Mean',
       'Reply_Length_Mean', 'ResponseLag_Mean', 'Investor_Tone',
       'Manager_Tone', 'Sentiment_Gap', 'Substantiveness_ML', 'Aggregate_Tone',
       'Newstone_t_30_t_3', 'Newsdummy_t_30_t_3', 'Size', 'Lev', 'ROA', 'BM',
       'TBQ', 'SUE', 'AdjEPS', 'Abs_DA', 'SOE', 'TOP1', 'DUAL', 'BoardSize_Ln',
       'INDEP', 'BALANCE', 'InsInvestorProp', 'AnaAttention',
       'Insider_Sell_Next30d', 'Repurchase_Next3M', 'Firm_Age_Ln', 'Imb_Noise',
       'Imb_Smart', 'Ln_MarginBuy', 'Dturn', 'Beta', 'Illiqd', 'ch4_mktrf',
       'ch4_SMB', 'ch4_VMG', 'ch4_PMO', 'q5_MKT', 'q5_ME', 'q5_IA', 'q5_ROE',
       'q5_EG', 'Mkt_RF', 'SMB', 'HML', 'Sentiment_Index'],
      dtype='object')